# 01 · Modal analysis of a cantilever beam

This notebook explains Step 2 of BeamTwin: how the beam **vibrates**.

The calculations live in the package (`src/beamtwin/fem/modal.py`) and are covered by automated tests.
This notebook only **imports** them, then shows and explains the results.

**Questions answered here**

1. What are the beam's natural frequencies, and do they match the textbook?
2. What do the mode shapes look like?
3. How many elements do we need for an accurate answer?

In [ ]:
# Reload the package automatically when a .py file changes
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from beamtwin.fem import cantilever_exact_frequencies, cantilever_modes

In [ ]:
# Chart style: light surface, quiet axes, thin lines
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # validated categorical order
MARKERS = ["o", "s", "^", "D"]                            # second channel, so colour is never alone

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 11,
})

# Save figures into docs/figures, wherever the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## The beam

A steel strip clamped at one end, the same one used throughout the project.

| Property | Symbol | Value |
|---|---|---|
| Length | L | 1.0 m |
| Width × thickness | b × h | 50 mm × 10 mm |
| Young's modulus | E | 200 GPa |
| Density | ρ | 7850 kg/m³ |

In [ ]:
b, h = 0.05, 0.01
beam = dict(E=200e9, I=b * h**3 / 12, rho=7850.0, A=b * h, length=1.0)

N_ELEMENTS = 32
freqs, shapes = cantilever_modes(**beam, n_elements=N_ELEMENTS, n_modes=4)
exact = cantilever_exact_frequencies(**beam)

print(f"{'Mode':>4}  {'FEM (Hz)':>10}  {'Textbook (Hz)':>13}  {'Error (%)':>9}")
for i, (f, fe) in enumerate(zip(freqs, exact), start=1):
    print(f"{i:>4}  {f:>10.3f}  {fe:>13.3f}  {abs(f - fe) / fe * 100:>9.4f}")

## 1. Mode shapes

Each mode is a pattern the beam can vibrate in on its own. Every shape is scaled so the tip moves by +1.

- **Mode 1** swings like a diving board, with no still points.
- **Mode 2** makes an S-shape, with one still point (a **node**) along the beam.
- **Mode n** has **n − 1** still points, and each mode vibrates faster than the one before.

Damage changes these shapes and frequencies slightly, which is what Step 4 will use to find it.

The solver only gives values at the **nodes**. Between nodes, each element bends along the same cubic curve
used to build its matrices (the *shape functions*). The helper below uses that curve to draw smooth shapes,
so the picture shows exactly what the model assumes.

In [ ]:
def smooth_shape(u, length, n_elements, points_per_element=12):
    # Draw a deflected shape using each element's cubic (Hermite) shape functions.
    L_e = length / n_elements
    s = np.linspace(0, 1, points_per_element, endpoint=False)   # position inside an element, 0..1
    N1 = 1 - 3 * s**2 + 2 * s**3            # weight of w at the left node
    N2 = L_e * (s - 2 * s**2 + s**3)        # weight of theta at the left node
    N3 = 3 * s**2 - 2 * s**3                # weight of w at the right node
    N4 = L_e * (-s**2 + s**3)               # weight of theta at the right node
    xs, ws = [], []
    for e in range(n_elements):
        w1, t1, w2, t2 = u[2 * e: 2 * e + 4]
        xs.append(e * L_e + s * L_e)
        ws.append(N1 * w1 + N2 * t1 + N3 * w2 + N4 * t2)
    xs.append([length]); ws.append([u[-2]])
    return np.concatenate(xs), np.concatenate(ws)


def zero_crossings(x, w):
    # Positions where the curve crosses zero (straight-line interpolation between samples).
    i = np.where(np.sign(w[1:-1]) != np.sign(w[2:]))[0] + 1     # skip the clamp, where w = 0
    return x[i] - w[i] * (x[i + 1] - x[i]) / (w[i + 1] - w[i])

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(7, 8), sharex=True)
for i, ax in enumerate(axes):
    xs, ws = smooth_shape(shapes[:, i], beam["length"], N_ELEMENTS)
    ax.axhline(0, color=MUTED, linewidth=1, linestyle="--")            # beam at rest
    ax.plot(xs, ws, color=SERIES[0])
    ax.plot([0, 0], [-1.3, 1.3], color=INK, linewidth=4)                # the clamp
    for x0 in zero_crossings(xs, ws):                                   # mark the still points
        ax.plot(x0, 0, "o", color=SURFACE, markeredgecolor=INK_2, markersize=7, zorder=3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_yticks([-1, 0, 1])
    ax.set_title(f"Mode {i + 1}  ·  {freqs[i]:.1f} Hz", loc="left")
    ax.set_ylabel("Deflection")

axes[-1].set_xlabel("Position along the beam (m)")
fig.suptitle("Cantilever mode shapes (clamped at x = 0, open circles mark still points)",
             x=0.02, ha="left", fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES / "mode_shapes.png", dpi=150)
plt.show()

## 2. How many elements do we need?

The exact mode shapes are not simple cubics, so the FEM answer is an approximation that improves as we
add elements. This is called a **convergence study**, and it is standard practice in FEM work.

The plot uses log scales on both axes. A straight line means the error falls at a steady rate as elements are added.

In [ ]:
n_list = [2, 4, 8, 16, 32]
errors = np.full((len(n_list), 4), np.nan)
for row, n in enumerate(n_list):
    n_modes = min(4, 2 * n)
    f, _ = cantilever_modes(**beam, n_elements=n, n_modes=n_modes)
    errors[row, :n_modes] = np.abs(f - exact[:n_modes]) / exact[:n_modes] * 100

# Below about 1e-6 % the "error" is the computer's rounding noise, not the FEM, so hide those points.
FLOOR = 1e-6
errors[errors < FLOOR] = np.nan

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.axhline(FLOOR, color=MUTED, linewidth=1, linestyle=":")
ax.annotate("approximate rounding floor: smaller errors are numerical noise", (1.25, FLOOR),
            xytext=(0, 4), textcoords="offset points", ha="left", va="bottom",
            color=MUTED, fontsize=8)
for i in range(4):
    ax.loglog(n_list, errors[:, i], color=SERIES[i], marker=MARKERS[i], markersize=8,
              markeredgecolor=SURFACE, markeredgewidth=1.5, label=f"Mode {i + 1}")
    ax.annotate(f"Mode {i + 1}", (n_list[0], errors[0, i]), xytext=(-10, 0),
                textcoords="offset points", va="center", ha="right", color=INK_2)

ax.set_xticks(n_list, [str(n) for n in n_list])
ax.minorticks_off()
ax.set_xlim(1.1, 45)
ax.set_xlabel("Number of elements")
ax.set_ylabel("Frequency error (%)")
ax.set_title("FEM frequency error vs. number of elements", loc="left")
ax.set_ylim(FLOOR / 5, 500)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "convergence.png", dpi=150)
plt.show()

## What this shows

- With 32 elements, all four frequencies are within **0.001 %** of the textbook values.
- **Higher modes need more elements**, because they have more bends to capture.
- Each doubling of the element count cuts the error by roughly 16×, the expected rate for cubic beam elements.
- Mode 1 drops below the **rounding floor** beyond 16 elements: its true error is then smaller than the noise
  from computer arithmetic (about 16 significant digits, reduced further as the matrices grow). Those points
  are hidden, because they would measure the computer, not the method.

For the surrogate and damage detection steps, we can choose the element count knowing exactly how much
accuracy we trade for speed.